<a href="https://colab.research.google.com/github/anastasiakalyashova/python-ai-AnastasiaKalyashova/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных о горах

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку географических данных с помощью pandas.

**Данные:**
- `mountains.csv` — информация о горах мира: название, координаты, высота и тип горных пород

**Что мы делаем:**
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файл с данными о горах в pandas DataFrame
3. Очищаем и анализируем структуру данных (координаты, высота, материалы)
4. Выполняем быструю валидацию данных и смотрим статистику


## 🐱 [1] Клонируем репозиторий курса в Colab

In [7]:
# 🐱 Шаг 1. Клонируем ваш репозиторий в Colab

import os

if not os.path.exists("python-ai-AnastasiaKalyashova"):
    !git clone -q https://github.com/anastasiakalyashova/python-ai-AnastasiaKalyashova.git

%cd python-ai-AnastasiaKalyashova

print("✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-AnastasiaKalyashova")

/content/python-ai-AnastasiaKalyashova/python-ai-AnastasiaKalyashova
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-AnastasiaKalyashova


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [8]:
# 🐱 Шаг 2A. Чтение CSV-файла с данными о горах

import pandas as pd

df_mountains = pd.read_csv("data/mountains.csv")

print("✅ Загружено строк в df_mountains:", len(df_mountains))
print("\nПервые 3 строки данных:")
print(df_mountains.head(3))

✅ Загружено строк в df_mountains: 4390

Первые 3 строки данных:
                              mountain mountainLabel  \
0  http://www.wikidata.org/entity/Q513   Джомолунгма   
1  http://www.wikidata.org/entity/Q513   Джомолунгма   
2  http://www.wikidata.org/entity/Q524       Везувий   

                  coordinates  elevation rockMaterialLabel  
0  Point(86.925 27.988055555)    8848.86     горная порода  
1  Point(86.925 27.988055555)    8848.86               лёд  
2    Point(14.42919 40.82261)    1281.00            Тефрит  


## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `mountain` с URL (ссылкой на объект Wikidata) — нам не нужна ссылка, нам достаточно названия горы.
- Столбцы `mountainLabel` и `rockMaterialLabel` содержат читаемые подписи (название горы и тип горной породы/материала).

В этом шаге мы:
- удалим столбец с URL Wikidata (`mountain`);
- переименуем `mountainLabel → mountain`, `rockMaterialLabel → rockMaterial`;
- приведём числовой столбец `elevation` (высота) к типу `float` (высота может содержать дробные значения, например 8848.86).

При приведении к числам мы используем:

- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(float)` — переводит столбец к дробному типу (для точного отображения высоты).

> ⚠️ **Важно:** если в ваших данных есть столбцы с URL Wikidata и столбцы вида `*Label`, этот шаг обязателен, чтобы получить аккуратные таблички для анализа.


In [9]:
# 🧹 Шаг 2B. Очистка и переименование столбцов

# Удаляем технический столбец с URL Wikidata
df_mountains = df_mountains.drop(columns=["mountain"])

# Переименовываем столбцы с читаемыми подписями
df_mountains = df_mountains.rename(columns={
    "mountainLabel": "mountain",
    "rockMaterialLabel": "rockMaterial"
})

# Приводим высоту к числовому типу (дробное число)
df_mountains["elevation"] = pd.to_numeric(
    df_mountains["elevation"], errors="coerce"
).fillna(0).astype(float)

print("✅ Данные очищены и готовы к анализу")
print("\nТекущие столбцы:", list(df_mountains.columns))
print("\nПример данных после очистки:")
print(df_mountains.head(3))

✅ Данные очищены и готовы к анализу

Текущие столбцы: ['mountain', 'coordinates', 'elevation', 'rockMaterial']

Пример данных после очистки:
      mountain                 coordinates  elevation   rockMaterial
0  Джомолунгма  Point(86.925 27.988055555)    8848.86  горная порода
1  Джомолунгма  Point(86.925 27.988055555)    8848.86            лёд
2      Везувий    Point(14.42919 40.82261)    1281.00         Тефрит


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame с данными о горах:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по высоте (`elevation`) — минимальная, максимальная, средняя высота и т.д.

Для удобства используем функцию `show_info(df, name)`, чтобы компактно вывести информацию о таблице.


In [10]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

# 🔍 Шаг 3. Обзор данных о горах

show_info(df_mountains, "Горы мира (df_mountains)")

print("\n📈 Статистика по высоте (elevation):")
print(df_mountains["elevation"].describe())

print("\n🪨 Распределение типов горных пород/материалов (rockMaterial):")
print(df_mountains["rockMaterial"].value_counts())


📊 Горы мира (df_mountains)
Размер: (4390, 4)
Столбцы: mountain, coordinates, elevation, rockMaterial

Первые строки:
      mountain                 coordinates  elevation   rockMaterial
0  Джомолунгма  Point(86.925 27.988055555)    8848.86  горная порода
1  Джомолунгма  Point(86.925 27.988055555)    8848.86            лёд
2      Везувий    Point(14.42919 40.82261)    1281.00         Тефрит
3      Монблан   Point(6.865 45.832777777)    4805.59         гранит
4      Монблан   Point(6.865 45.832777777)    4805.59          гнейс

📈 Статистика по высоте (elevation):
count     4390.000000
mean      1581.164787
std       1300.251027
min        -39.000000
25%        692.000000
50%       1233.900000
75%       2283.500000
max      16390.000000
Name: elevation, dtype: float64

🪨 Распределение типов горных пород/материалов (rockMaterial):
rockMaterial
известняк         834
песчаник          518
гранит            346
Мергель           301
Конгломерат       289
                 ... 
green tuff     

In [11]:
# То же самое, но в километрах (км) для удобства восприятия
stats_km = (df_mountains["elevation"] / 1000).describe().round(2)
print("\n📈 Статистика по высоте в км:")
print(stats_km)



📈 Статистика по высоте в км:
count    4390.00
mean        1.58
std         1.30
min        -0.04
25%         0.69
50%         1.23
75%         2.28
max        16.39
Name: elevation, dtype: float64


## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** фильмов, стран и жанров есть в данных;
- **какие страны встречаются чаще всего** (Топ‑5 по числу строк);
- **какие жанры самые популярные** (Топ‑10 по числу строк).

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому  
`df_genre["country"].value_counts().head()` даёт **Топ‑5 стран по числу записей**.

In [ ]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных")

# Датасет 1: бюджеты
print("\nУникальных мультфильмов в df_cost:", df_cost["film"].nunique())
print("Диапазон бюджетов (млн $):",
      df_cost["capital_cost"].min() / 1e6, "—", df_cost["capital_cost"].max() / 1e6)

# Датасет 2: жанры, страны, длительность
print("\nУникальных мультфильмов в df_genre:", df_genre["film"].nunique())
print("Уникальных стран:", df_genre["country"].nunique())
print("Уникальных жанров:", df_genre["genre"].nunique())

print("\nТоп-5 стран по числу записей:")
print(df_genre["country"].value_counts().head())

print("\nТоп-10 жанров:")
print(df_genre["genre"].value_counts().head(10))

🔍 Быстрая проверка данных

Уникальных мультфильмов в df_cost: 417
Диапазон бюджетов (млн $): 9e-06 — 2800.0

Уникальных мультфильмов в df_genre: 2273
Уникальных стран: 86
Уникальных жанров: 240

Топ-5 стран по числу записей:
country
США        2836
Франция     527
СССР        488
Дания       338
Россия      334
Name: count, dtype: int64

Топ-10 жанров:
genre
приключенческий фильм          970
комедийный фильм               833
фэнтезийный фильм              778
семейный фильм                 735
детский фильм                  627
мультфильм                     456
музыкальный фильм              304
драматический фильм            256
боевик                         249
научно-фантастический фильм    220
Name: count, dtype: int64


## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Посмотрели базовую статистику по бюджету (`capital_cost`)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨